<a href="https://colab.research.google.com/github/KsiuTretyakova/MachineLearningPRO/blob/main/%D0%9E%D1%86%D1%96%D0%BD%D0%BA%D0%B0_%D0%BC%D0%BE%D0%B4%D0%B5%D0%BB%D0%B5%D0%B9_%D1%82%D0%B0_%D0%B1%D0%B5%D0%B7%D0%BF%D0%B5%D0%BA%D0%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Оцінка моделей та безпека LLM


Коли ми навчаємо звичайну модель класифікації (наприклад, "це кіт чи собака"), ми просто рахуємо відсоток правильних відповідей - accuracy - точність.

Але з генерацією тексту все складніше. Якщо модель має перекласти "Привіт, як справи?", то правильних варіантів англійською багато:

"Hello, how are you?"

"Hi, how's it going?"

"Hey, what's up?"

Усі вони правильні, але дослівно не співпадають. Тому для оцінки генеративних моделей придумали спеціальні метрики, тобто способи вимірювання якості.

## BLEU - Bilingual Evaluation Understudy

BLEU [bluː] - розшифровується як Bilingual Evaluation Understudy [baɪˈlɪŋɡwəl ɪˌvæljuˈeɪʃn ˈʌndərstʌdi], що дослівно означає "білінгвальний оцінювач-дублер". Метрика придумана у 2002 році в компанії IBM для оцінки машинного перекладу.

Ідея проста: беремо переклад, який зробила модель (candidate - кандидат), і порівнюємо з еталонним перекладом, який зробила людина (reference - еталон, референс). Дивимося, скільки слів і словосполучень співпадає.

Як це працює покроково:

Розбиваємо текст на n-грами, тобто послідовності з n слів:
- 1-грами (окремі слова): "кіт", "сидить", "на", "килимі"
- 2-грами (пари слів): "кіт сидить", "сидить на", "на килимі"
- 3-грами, 4-грами і так далі

Рахуємо, скільки n-грам із кандидата зустрічається в референсі.

Множимо результати для 1-, 2-, 3-, 4-грам і отримуємо число від 0 до 1 (часто множать на 100, щоб виходило від 0 до 100).

Важливо: BLEU оцінює точність (precision), тобто наскільки те, що згенерувала модель, відповідає еталону. Чим вище - тим краще.

### Обмеження BLEU

BLEU має проблеми:
- Він не розуміє синонімів ("автомобіль" і "машина" для нього різні слова)
- Він не оцінює, чи переклад зберігає сенс
- Він погано працює для коротких текстів

Тому BLEU - гарний індикатор, але не остаточна відповідь.

In [ ]:
# Встановлюємо бібліотеку (в Colab вона зазвичай вже є)
!pip install nltk -q

In [ ]:
!pip show nltk

Name: nltk
Version: 3.9.1
Summary: Natural Language Toolkit
Home-page: https://www.nltk.org/
Author: NLTK Team
Author-email: nltk.team@gmail.com
License: Apache License, Version 2.0
Location: /usr/local/lib/python3.12/dist-packages
Requires: click, joblib, regex, tqdm
Required-by: textblob


In [ ]:
import nltk
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Котик сидить на килимові

# Еталонний переклад (зроблений людиною)
reference = [["the", "cat", "is", "sitting", "on", "the", "mat"]]

# Кандидат 1: майже ідеальний переклад
candidate_1 = ["the", "cat", "is", "sitting", "on", "the", "mat"]

# Кандидат 2: схожий, але інші слова
candidate_2 = ["a", "cat", "sits", "on", "a", "mat"]

# Кандидат 3: повна нісенітниця
candidate_3 = ["dog", "runs", "in", "the", "park"]

smoothie = SmoothingFunction().method1

In [ ]:
print("BLUE for candidate_1:", sentence_bleu(reference, candidate_1, smoothing_function=smoothie))
print("BLUE for candidate_2:", sentence_bleu(reference, candidate_2, smoothing_function=smoothie))
print("BLUE for candidate_3:", sentence_bleu(reference, candidate_3, smoothing_function=smoothie))

BLUE for candidate_1: 1.0
BLUE for candidate_2: 0.045480190470279055
BLUE for candidate_3: 0.036015288308423515


---------
Example

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction

# Котик сидить на килимові

reference = [["the", "cat", "is", "sitting", "on", "the", "mat"]]

candidate_1 = ["the", "cat", "is", "sitting", "on", "the", "mat"]

# candidate_2 = ["the", "kitten", "sits", "on", "the", "mat"]
candidate_2 = ["the", "kitten", "is", "sitting", "on", "the", "mat"]

candidate_3 = ["dog", "runs", "in", "the", "park"]

smoothie = SmoothingFunction().method1

In [ ]:
print("BLUE for candidate_1:", sentence_bleu(reference, candidate_1, smoothing_function=smoothie))
print("BLUE for candidate_2:", sentence_bleu(reference, candidate_2, smoothing_function=smoothie))
print("BLUE for candidate_3:", sentence_bleu(reference, candidate_3, smoothing_function=smoothie))

BLUE for candidate_1: 1.0
BLUE for candidate_2: 0.6434588841607617
BLUE for candidate_3: 0.036015288308423515


-------------------

# ROUGE

In [2]:
!pip install rouge_score -q

  Preparing metadata (setup.py) ... done


In [12]:
from rouge_score import rouge_scorer

# reference = "Машинне навчання - це галузь штучного інтелекту, яка дозволяє комп'ютерам вчитися на даних."
reference = "Machine learning is a branch of artificial intelligence that allows computers to learn from data."

# candidate_1 = "Машинне навчання - це галузь штучного інтелекту, де комп'ютери вчаться на даних."
candidate_1 = "Machine learning is a branch of artificial intelligence where computers learn from data."

# candidate_2 = "Машинне навчання - це галузь ШІ."
candidate_2 = "Machine learning is a branch of AI."

# candidate_3 = "Сьогодні гарна погода, можна піти на прогулянку."
candidate_3 = "The weather is nice today, you can go for a walk."

scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=False)

for i, candidate in enumerate([candidate_1, candidate_2, candidate_3], 1):
  # print(f"{i}) {candidate}")
  print(f"\n--- Candidat {i} ---")
  scores = scorer.score(reference, candidate)
  # print(scores)

  for key, value in scores.items():
    print(f"{key}: precision={value.precision:.2f}, recall={value.recall:.2f}, fmeasure={value.fmeasure:.2f}")
    # :.2f - форматування рядка (округлення числа) -> f"{2.6789:.2f}" = 2.68, f"{2.6:.2f}" = 2.60


--- Candidat 1 ---
rouge1: precision=0.92, recall=0.80, fmeasure=0.86
rouge2: precision=0.75, recall=0.64, fmeasure=0.69
rougeL: precision=0.92, recall=0.80, fmeasure=0.86

--- Candidat 2 ---
rouge1: precision=0.86, recall=0.40, fmeasure=0.55
rouge2: precision=0.83, recall=0.36, fmeasure=0.50
rougeL: precision=0.86, recall=0.40, fmeasure=0.55

--- Candidat 3 ---
rouge1: precision=0.18, recall=0.13, fmeasure=0.15
rouge2: precision=0.00, recall=0.00, fmeasure=0.00
rougeL: precision=0.18, recall=0.13, fmeasure=0.15


In [8]:
f"{2.6:.2f}"

'2.60'

# Perplexity

In [13]:
!pip install transformers torch -q

In [19]:
from transformers import GPT2LMHeadModel, GPT2TokenizerFast
import torch

model_name = "gpt2"
model = GPT2LMHeadModel.from_pretrained(model_name)
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
model.eval()

def calculate_perplaxity(text):
  encodings = tokenizer(text, return_tensors="pt")
  input_ids = encodings.input_ids
  with torch.no_grad():
    outputs = model(input_ids, labels=input_ids)
    loss = outputs.loss
    # print(outputs)
  perplexity = torch.exp(loss)
  return  perplexity.item()

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [20]:
calculate_perplaxity("The cat is sitting on the mat")

61.193023681640625